# WP3: Sea Ice / Open Water Classification

NDSI thresholding, Random Forest classification (S2), SAR VV thresholding (S1), and per-pixel fusion.

In [ ]:
import ee
import geemap
import sys
sys.path.insert(0, '..')
from src.utils import load_aoi, get_gee_project
from src.preprocessing import preprocess_s2, preprocess_s1
from src.classification import classify_ndsi, classify_sar, train_random_forest, classify_rf, fuse_classifications

ee.Initialize(project=get_gee_project())
aoi = load_aoi()

## 3.1 NDSI threshold classification

In [ ]:
START, END = '2019-01-01', '2024-12-31'

s2 = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi).filterDate(START, END)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80))
    .map(preprocess_s2)
)
s2_classified = s2.map(classify_ndsi)

## 3.2 Random Forest classifier

> Requires manually digitised training polygons as a GEE FeatureCollection (asset).

In [ ]:
# TODO: replace with your GEE asset path
# training_fc = ee.FeatureCollection('users/your-username/sermilik_training')
# rf_bands = ['B2', 'B3', 'B4', 'B8', 'B11', 'NDSI']
# classifier = train_random_forest(training_fc, rf_bands)
# s2_rf = s2.map(lambda img: classify_rf(img, classifier, rf_bands))

## 3.3 SAR threshold classification

In [ ]:
s1 = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(aoi).filterDate(START, END)
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .select(['VV', 'VH'])
    .map(preprocess_s1)
)
s1_classified = s1.map(classify_sar)

## 3.4 Fusion — TODO

Fuse per acquisition date using `fuse_classifications()` from `src/classification.py`.

In [ ]:
# Implement temporal join and pixel-level fusion here

## 3.5 Visualise

In [ ]:
Map = geemap.Map()
Map.centerObject(aoi, zoom=9)
Map.addLayer(
    s2_classified.first().select('ice_ndsi'),
    {'min': 0, 'max': 1, 'palette': ['#1a6faf', 'white']},
    'NDSI ice (first image)'
)
Map